# FinBase support assistant - my build

my notes and code, in the order i built it. run the cells top to bottom.
needs: pypdf, numpy, sentence-transformers (i installed them with %pip in a cell above this).


In [ ]:
import os, re, json, time, random
import numpy as np
from pypdf import PdfReader

# i keep the 6 pdfs in data/raw, next to this notebook
RAW = "data/raw"
if not os.path.isdir(RAW):
    # when i started the pdfs were lying loose in the folder, this also works
    RAW = "."

WORK = "work"
os.makedirs(WORK, exist_ok=True)
print("reading pdfs from:", RAW)
print(sorted([f for f in os.listdir(RAW) if f.endswith(".pdf")]))

## step 1 - pull the text out of the pdfs


In [ ]:
texts = {}
for name in sorted(os.listdir(RAW)):
    if not name.endswith(".pdf"):
        continue
    reader = PdfReader(os.path.join(RAW, name))
    text = ""
    for page in reader.pages:
        text = text + (page.extract_text() or "") + "\n"
    doc = name[:-4]                      # strip ".pdf"
    texts[doc] = text
    print(doc, "->", len(text), "chars,", len(reader.pages), "pages")

i opened work/sample_5.txt and wrote down what is wrong with the raw text:
- every ₹ came out as ■
- bullet points came out as a weird invisible character
- there is a table of contents full of links like ](#section-4-2)
- one sentence is often broken across two lines


## step 2 - fix those 4 problems


In [ ]:
def clean(text):
    counts = {}

    # problem 1: table of contents lines are useless for search, delete them
    kept = []
    removed = 0
    for line in text.split("\n"):
        if "](#section-" in line:
            removed = removed + 1
        else:
            kept.append(line)
    text = "\n".join(kept)
    counts["toc_removed"] = removed

    # problem 2: broken characters
    counts["rupees_fixed"] = text.count("■")
    text = text.replace("■", "₹")
    counts["bullets_fixed"] = text.count("\x7f")
    text = text.replace("\x7f", "•")

    # problem 3: join lines that are actually one sentence
    out = []
    joined = 0
    for line in text.split("\n"):
        line = line.strip()
        if line == "":
            continue
        # these lines always start something new, never join them upwards
        starts_new = (re.match(r"Section \d", line) or line.startswith("•")
                      or re.match(r"Q\d+:", line) or re.match(r"[A-Za-z ]{3,40}:", line))
        prev_heading = len(out) > 0 and (re.match(r"Section \d", out[-1]) or re.match(r"Q\d+:", out[-1]))
        if len(out) > 0 and not starts_new and not prev_heading:
            last = out[-1]
            if not last.endswith((".", "!", "?", ":")):
                out[-1] = last + " " + line     # continuation of the sentence
                joined = joined + 1
                continue
        out.append(line)
    counts["lines_joined"] = joined
    return "\n".join(out), counts

# try it on one doc first
sample, counts = clean(texts["sample_5"])
print("sample_5:", counts)
print(sample[:400])

In [ ]:
# now all six
cleaned = {}
totals = {}
for doc in texts:
    cleaned[doc], totals[doc] = clean(texts[doc])
    f = open(WORK + "/clean_" + doc + ".txt", "w", encoding="utf-8")
    f.write(cleaned[doc])
    f.close()

print("totals for the whole corpus:")
for key in ("toc_removed", "bullets_fixed", "rupees_fixed", "lines_joined"):
    total = 0
    for doc in totals:
        total = total + totals[doc][key]
    print(" ", key, "=", total)

## step 3 - chunking

my strategy: the documents already have structure (sections + a 100-question FAQ
at the end of each doc), so i split on that. long sections get cut into ~180 word
pieces with 30 words repeated so a sentence never gets sliced in half.


In [ ]:
heading_re = re.compile(r"^(Section \d+(\.\d+)? *:|(\d+\.\d+) [A-Z])")
faq_re = re.compile(r"^Q(\d+):")

def split_blocks(text):
    blocks = []
    head = None
    lines = []
    for line in text.split("\n"):
        if heading_re.match(line) or faq_re.match(line):
            if len(lines) > 0:
                blocks.append({"head": head, "lines": lines})
            head = line
            lines = [line]
        else:
            lines.append(line)
    if len(lines) > 0:
        blocks.append({"head": head, "lines": lines})
    return blocks

def make_chunks(text):
    out = []
    for b in split_blocks(text):
        body = " ".join(b["lines"]).strip()
        words = body.split()
        is_faq = faq_re.match(b["head"] or "") is not None
        if not is_faq and len(words) > 180:
            start = 0
            part = 1
            while start < len(words):
                piece = words[start:start + 180]
                out.append({"text": " ".join(piece), "head": b["head"],
                            "part": part, "faq": False})
                start = start + 150          # 180 - 30 words of overlap
                part = part + 1
        else:
            out.append({"text": body, "head": b["head"], "part": 0, "faq": is_faq})
    return out

In [ ]:
chunks = []
for doc in cleaned:
    title = cleaned[doc].split("\n")[0].strip()     # first line = document title
    for c in make_chunks(cleaned[doc]):
        head = (c["head"] or "").strip()
        if c["faq"]:
            qnum = head.split(":")[0].strip()[1:]   # "046"
            section = "FAQ-" + qnum.zfill(3)
            citation = title + " — FAQ Q" + qnum.zfill(3)
        else:
            if head.startswith("Section"):
                section = head.replace("Section", "").split(":")[0].strip()
            elif re.match(r"\d+\.\d+", head):
                section = head.split(" ")[0]
            else:
                section = "0"
            citation = title + " — " + (head[:70] if head != "" else "Preamble")
        if c["part"] > 1:
            citation = citation + " (part " + str(c["part"]) + ")"
        chunks.append({"chunk_id": doc + "::" + section + "::" + str(len(chunks)),
                       "doc": doc, "section": section, "citation": citation,
                       "text": c["text"], "faq": c["faq"]})

f = open(WORK + "/chunks.jsonl", "w", encoding="utf-8")
for c in chunks:
    f.write(json.dumps(c, ensure_ascii=False) + "\n")
f.close()

faq_count = 0
for c in chunks:
    if c["faq"]:
        faq_count = faq_count + 1
print("total chunks:", len(chunks), "| faq chunks:", faq_count)

# eyeball 5 random chunks: do they make sense on their own?
for c in random.sample(chunks, 5):
    print("\n", c["citation"])
    print(" ", c["text"][:150])

## step 4 - embeddings

i use all-MiniLM-L6-v2: small, runs on my laptop cpu in ~30 seconds,
384 numbers per chunk, and fine for short policy sentences.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

t0 = time.time()
vectors = model.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)
np.save(WORK + "/vectors.npy", vectors)     # row i = chunks[i]
print("embedded", len(chunks), "chunks in", round(time.time() - t0, 1), "seconds")
print("shape:", vectors.shape)

## step 5 - search

cosine similarity (dot product works because the vectors are normalised),
plus a small bonus when the section title contains words from the question.
i added that bonus after i saw 'complaint' questions dragging in the wrong doc.


In [ ]:
def search(query, k=5):
    qv = model.encode([query], normalize_embeddings=True)[0]
    scores = list(vectors @ qv)
    qwords = list(set(re.findall(r"[a-z]{4,}", query.lower())))
    for i in range(len(chunks)):
        bonus = 0
        for w in qwords:
            if w in chunks[i]["citation"].lower():
                bonus = bonus + 0.06
        scores[i] = scores[i] + bonus
    order = np.argsort(-np.array(scores))
    hits = []
    for i in order[:k]:
        hits.append((float(scores[i]), chunks[i]))
    return hits

for q in ["What is the foreclosure charge on a personal loan?",
          "What is the minimum balance for a savings account?",
          "How do I raise a dispute for an unauthorized transaction?"]:
    print("\nQ:", q)
    for score, c in search(q, 3):
        print(" ", round(score, 3), c["citation"][:80])

## step 6 - knowing what the kb does NOT contain

trick: collect every word that appears anywhere in the corpus. if a question
uses a word the corpus has never seen (home, nri, locker...), the kb cannot
possibly answer it. common english words are ignored, i kept adding to that
list every time a normal question got flagged by mistake.


In [ ]:
vocab = set()
for c in chunks:
    for w in re.findall(r"[a-z]{3,}", c["text"].lower()):
        vocab.add(w)

STOP = """the a an of for to in on at my i how do does can could you me and or are
was were be been being this that these those with without about from into over under
after before during if then than so such between across per any all each other more
most some no not only own same too very just don should now there here when where why
which who whom whose will would shall may might must have has had having it its please
tell show give need want get got make made use used using also long take quickly keep
ready complaint close open work much many days day time times money account rate
charge fee fees one two three""".split()

def uncovered(query):
    words = set(re.findall(r"[a-z]{3,}", query.lower())) - set(STOP)
    missing = []
    for w in sorted(words):
        if w not in vocab:
            missing.append(w)
    return missing

for q in ["What is the interest rate on a home loan?",
          "What is the FD rate for NRI customers?",
          "What is the foreclosure charge on a personal loan?"]:
    print(uncovered(q), "<-", q)

## step 7 - answering + refusing

answer = the 2 sentences from each top chunk that share the most words with the
question, quoted as-is (i never let the model invent text). two safety rules:
1. refuse when 3+ words are unknown to the corpus, or when the best match is weak
2. if 1-2 words are unknown (a product we don't have), say exactly what is
   missing and then show the closest related info


In [ ]:
REFUSAL = "I could not find this information in the FinBase knowledge base."

def get_sentences(text):
    # drop the "(Operational case 12)" style annotations, then split on . ! ?
    text = re.sub(r"\s*\((Operational|Payment|Service|KYC|Card|Loan|Wealth|Savings|UPI)[a-z ]*case \d+\)", "", text)
    text = text.replace("•", "").strip()
    out = []
    for s in re.split(r"(?<=[.?!]) ", text):
        s = s.strip()
        if len(s) > 25 and not re.match(r"(Section \d|Q\d+:) [^.]{0,60}$", s):
            out.append(s)
    return out

def answer(query):
    hits = search(query, 5)
    missing = uncovered(query)

    # rule 1: off-topic, or nothing relevant retrieved -> refuse
    if len(missing) >= 3 or hits[0][0] < 0.35:
        return {"answer": REFUSAL, "citations": []}

    # rule 2: mentions a product the bank does not have -> say exactly that,
    # then show the closest thing we do have
    lead = ""
    if len(missing) > 0:
        lead = 'I could not find information about "' + ", ".join(missing) + \
               '" in the FinBase knowledge base. The closest related information available is: '

    qwords = set(re.findall(r"[a-z0-9]+", query.lower()))
    pieces = []
    citations = []
    rank = 0
    for score, ch in hits:
        rank = rank + 1
        sents = get_sentences(ch["text"])
        if len(sents) == 0:
            continue
        # how many question words does each sentence contain?
        overlap = []
        for i in range(len(sents)):
            sword = set(re.findall(r"[a-z0-9]+", sents[i].lower()))
            overlap.append((len(qwords.intersection(sword)), i))
        overlap.sort(key=lambda x: -x[0])
        if overlap[0][0] == 0:
            continue                        # nothing in common, skip this chunk
        top = [overlap[0][1]]
        if len(overlap) > 1:
            top.append(overlap[1][1])
        top.sort()                          # keep reading order
        piece = ""
        for i in top:
            piece = piece + sents[i] + " "
        pieces.append(piece.strip() + " [S" + str(rank) + "]")
        citations.append({"label": ch["citation"], "snippet": ch["text"][:150]})

    if len(pieces) == 0:
        return {"answer": REFUSAL, "citations": []}
    body = "Based on the FinBase knowledge base:\n" + "\n".join(pieces)
    if lead != "":
        body = lead + body
    return {"answer": body, "citations": citations}

## step 8 - try it (expected: 3 answered, 3 disclosed, 1 refused)


In [ ]:
grid = ["What is the foreclosure charge on a personal loan?",
        "What is the minimum balance for a savings account?",
        "What do I need to keep ready for video KYC?",
        "What are the charges for a bank locker facility?",
        "What is the interest rate on a home loan?",
        "What is the FD interest rate for NRI customers?",
        "What is the capital of France?"]

for q in grid:
    r = answer(q)
    if r["answer"] == REFUSAL:
        tag = "REFUSED"
    elif r["answer"].startswith("I could not find information about"):
        tag = "DISCLOSED"
    else:
        tag = "ANSWERED (" + str(len(r["citations"])) + " sources)"
    print("\nQ:", q)
    print(" [" + tag + "]", r["answer"][:220])
    if len(r["citations"]) > 0:
        print("  source:", r["citations"][0]["label"])

## step 9 - my own test set (golden set)

write questions YOURSELF from the real text: 30-35 answerable across all 6 docs
+ about 10 traps (products the bank does not have: home loan, car loan, locker,
NRI, Titanium card, salary account, cheque book, demat, forex card...).
use peek() to read the facts first. gold_section must match my section numbers.


In [ ]:
def peek(keyword, n=2):
    found = 0
    for c in chunks:
        if keyword.lower() in c["text"].lower():
            print("\n", c["citation"])
            print(" ", c["text"][:400])
            found = found + 1
            if found == n:
                return

peek("foreclosure")

golden = [
  {"id": "A1", "q": "What is the foreclosure charge on a personal loan?",
   "answerable": True, "gold_doc": "sample_5", "gold_section": "6.2",
   "gold_facts": ["3%", "24 months"]},
  {"id": "T1", "q": "What are the charges for a bank locker facility?",
   "answerable": False, "gold_doc": None, "gold_section": None, "gold_facts": []},
  # ... keep writing here until you have 30+ answerable and ~10 traps
]

f = open("golden_set.json", "w", encoding="utf-8")
json.dump(golden, f, indent=1, ensure_ascii=False)
f.close()
print("items:", len(golden))

## step 10 - evaluation


In [ ]:
def fact_coverage(ans, facts):
    # ignore ₹ and commas so "₹4,999" and "4999" both count
    a = ans.replace("₹", "").replace(",", "")
    if len(facts) == 0:
        return 1.0
    hit = 0
    for fact in facts:
        if fact.replace("₹", "").replace(",", "") in a:
            hit = hit + 1
    return hit / len(facts)

rows = []
for g in golden:
    hits = search(g["q"], 5)
    sections = []
    for score, c in hits:
        sections.append(c["section"])
    r = answer(g["q"])
    refused = r["answer"] == REFUSAL
    disclosed = r["answer"].startswith("I could not find information about")
    row = {"id": g["id"], "answerable": g["answerable"], "refused": refused,
           "disclosed": disclosed,
           "hit@1": g["answerable"] and sections[0] == g["gold_section"],
           "hit@3": g["answerable"] and (g["gold_section"] in sections[:3]),
           "hit@5": g["answerable"] and (g["gold_section"] in sections[:5]),
           "coverage": fact_coverage(r["answer"], g["gold_facts"]) if g["answerable"] else None}
    rows.append(row)

good = [r for r in rows if r["answerable"]]
traps = [r for r in rows if not r["answerable"]]

def avg(rows, key):
    total = 0
    for r in rows:
        if r[key]:
            total = total + 1
    return round(total / len(rows), 3)

lines = []
lines.append("# evaluation report (my numbers)")
lines.append("")
lines.append("answerable: " + str(len(good)) + " | traps: " + str(len(traps)))
lines.append("")
lines.append("retrieval on answerable questions:")
lines.append("- hit@1: " + str(avg(good, "hit@1")))
lines.append("- hit@3: " + str(avg(good, "hit@3")))
lines.append("- hit@5: " + str(avg(good, "hit@5")))
lines.append("")
cov = 0
for r in good:
    cov = cov + r["coverage"]
lines.append("answer fact coverage: " + str(round(cov / len(good), 3)))
lines.append("")
handled = 0
for r in traps:
    if r["refused"] or r["disclosed"]:
        handled = handled + 1
lines.append("traps refused or disclosed: " + str(handled) + "/" + str(len(traps)))
lines.append("")
lines.append("failures:")
for r in rows:
    if r["answerable"] and (not r["hit@3"] or (r["coverage"] or 0) < 1.0):
        lines.append("- " + r["id"] + ": hit@3=" + str(r["hit@3"]) + ", coverage=" + str(r["coverage"]))
    if not r["answerable"] and not (r["refused"] or r["disclosed"]):
        lines.append("- " + r["id"] + " (trap): leaked, answered instead of refusing")

report = "\n".join(lines)
f = open("eval_report.md", "w", encoding="utf-8")
f.write(report)
f.close()
print(report)

## step 11 - the numbers i quote in readme + video


In [ ]:
glyphs = 0
toc = 0
joins = 0
for doc in totals:
    glyphs = glyphs + totals[doc]["bullets_fixed"] + totals[doc]["rupees_fixed"]
    toc = toc + totals[doc]["toc_removed"]
    joins = joins + totals[doc]["lines_joined"]

numbers = {
    "chunks": len(chunks),
    "faq_chunks": faq_count,
    "toc_lines_removed": toc,
    "broken_characters_fixed": glyphs,
    "lines_joined": joins,
    "hit@1": avg(good, "hit@1"),
    "hit@5": avg(good, "hit@5"),
    "fact_coverage": round(cov / len(good), 3),
    "traps_handled": str(handled) + "/" + str(len(traps)),
}
f = open(WORK + "/final_numbers.json", "w")
f.write(json.dumps(numbers, indent=1))
f.close()
print(json.dumps(numbers, indent=1))